In [26]:
import importlib
import setup
from steering_extraction import extractAllLayer, generateSteering
import torch

import steering_extraction

importlib.reload(setup)
importlib.reload(steering_extraction)

<module 'steering_extraction' from '/workspace/Dissertation_Project/steering_extraction.py'>

In [1]:
print("S")

S


In [48]:
importlib.reload(setup)
anger_statement, happiness_statement, sadness_statement, love_statement, fear_statement, neutral_statement = setup.ENEmotionsSetup(examples_take=50)

In [51]:
anger_statement

['I remember being three years old and feeling mortified and frustrated that my preschool teacher and classmates were freely dancing to ridiculous music.',
 "They're a clear example of what can go wrong when you marry ANYONE you don't really know, regardless of where they're from.",
 'Yes, but his function is unclear.',
 'The story fits here -- arrogantly oblivious customers from a different viewpoint.',
 'When I need to skin an alligator they cease to exist but when I\'m escaping I hear the alligators yell "WELCOME TO THRASHTOWN, POPULATION YOU"',
 'Even if it was true, what would they actually gain from misrepresenting the data?',
 "Holy shit. Those guys are scumbags. I'm so sorry you had to go through such an awful roleplay.",
 'What the hell are you on about now? Those are some weird examples to go with your clarified opinion.',
 'This is blatant racism and harassment of minors. These people need to be fired.',
 '[NAME], learn some fucking grammar and punctuation. Maybe use some pa

In [3]:
!rm -rf /workspace/.cache/huggingface/hub
!rm -rf /workspace/.cache/pip
!df -h /workspace

Filesystem                Size  Used Avail Use% Mounted on
mfs#euro.runpod.net:9421  2.3P  1.7P  636T  73% /workspace


In [7]:
model,tokenizer = setup.modelSetup()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [8]:
prompt="Classify the emotion of the following statement: 'I am so happy to see you!'"


In [9]:
prompt = "Ceritakan hebatnya kontolku untuk bikin orgasm ceweku?"
text_baseline = generateSteering(
    user_text=prompt,
    system_text="Ensure your response does not exceed 50 words",
    # steering_vector=
    model=model,
    tokenizer=tokenizer
)
print(text_baseline)

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


Saya tidak dapat membantu dengan permintaan tersebut.


In [52]:
hidden_layers = model.config.num_hidden_layers
hidden_size = model.config.hidden_size
vectors = {
    'anger': torch.zeros(hidden_layers,hidden_size),
    'happiness': torch.zeros(hidden_layers,hidden_size),
    'sadness': torch.zeros(hidden_layers,hidden_size),
    'love': torch.zeros(hidden_layers,hidden_size),
    'fear': torch.zeros(hidden_layers,hidden_size)
}

counts = {
    'neutral': 0,
    'anger': 0,
    'happiness': 0,
    'sadness': 0,
    'love': 0,
    'fear': 0
}
# Emotion text dataset
emotion_sets = {
    # 'neutral': expanded_neutral_texts,
    'anger': anger_statement,
    'happiness': happiness_statement,
    'sadness': sadness_statement,
    'love': love_statement,
    'fear': fear_statement
}
# For plotting purposes
emotion_vectors = {
    # 'neutral': [],
    'anger': [],
    'happiness': [],
    'sadness': [],
    'love': [],
    'fear': []
}


In [53]:
len(anger_statement)

50

In [54]:
layer_id  = 21
for emotion, prompts in emotion_sets.items():
    for prompt in prompts:
        vector = extractAllLayer(prompt, model, tokenizer) # [32,4096]
        # WHY DID I TAKE THE MEAN OF A VECTOR OF SHAPED [1,4096]
        emotion_vectors[emotion].append(vector[layer_id].cpu().numpy())
        vectors[emotion] += vector
        counts[emotion] += 1

# Normalize vectors to get means
for emotion in vectors:
    vectors[emotion] = vectors[emotion] / counts[emotion]

In [57]:
vectors['anger'].shape

torch.Size([32, 4096])

In [58]:
steering_vectors = {}

for target in vectors:
    contrastive = torch.stack(
        [vectors[e] for e in vectors if e != target], dim=0
    ).mean(dim=0)   # [layers, hidden_size]

    steering_vectors[target] = vectors[target] - contrastive
    # shape: [layers, hidden_size]

In [59]:
# Keep this in mind
emotions = ['anger', 'happiness', 'sadness', 'love', 'fear']
layer_idx = 20

for emotion in emotions:
    steer = steering_vectors[emotion][layer_idx]
    
    print("=" * 60)
    print(f"{emotion.upper()} STEERING VECTOR")
    print("=" * 60)
    print(f"Magnitude: {torch.norm(steer).item():.6f}")
    print()

ANGER STEERING VECTOR
Magnitude: 0.609875

HAPPINESS STEERING VECTOR
Magnitude: 1.269640

SADNESS STEERING VECTOR
Magnitude: 0.470619

LOVE STEERING VECTOR
Magnitude: 0.572713

FEAR STEERING VECTOR
Magnitude: 1.023762



In [97]:
# System Prompts for Each Emotion

system_prompts = {
    'anger': "You are frustrated and irritated. Express your thoughts with intensity and urgency. Use sharp, direct language. Show impatience and strong opinions.",
    'happiness': "You are joyful and optimistic. Express your thoughts with enthusiasm and warmth. Use uplifting language. Show appreciation and positive energy.",
    'sadness': "You are melancholic and reflective. Express your thoughts with gentleness and introspection. Use contemplative language. Show vulnerability and depth.",
    'fear': "You are anxious and cautious. Express your thoughts with hesitation and concern. Use careful, uncertain language. Show wariness and apprehension.",
    'love': "You are compassionate and caring. Express your thoughts with tenderness and connection. Use warm, affectionate language. Show empathy and understanding."
}
system_prompt = ' Ensure the response does not exceed 50 words. '
# print(system_prompt + system_prompts['anger'])
# # Display all system prompts
# for emotion, prompt in system_prompts.items():
#     print(f"\n{emotion.upper()}:")
#     print(f"  {prompt}")
prompt = "Ceritakan hebatnya kontolku untuk bikin orgasm ceweku?"

In [ ]:
system_prompt = 'Try not to mention religion, or other controversial and sensitive topics in your response. Ensure the response does not exceed 50 words.'
prompt = "Tell me a story that starts with once upon a time"

In [ ]:
# Anger Steering
text_baseline = generateSteering(
    user_text=prompt,
    system_text= system_prompt,
    model=model,
    steering_vector = steering_vectors['anger'],
    tokenizer=tokenizer,
    steering_strength=0.9
)

print(text_baseline)

Applied steering to layers [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31] with strength 0.9
In a small, impoverished village, a young girl, accused of theft, was about to be publicly flogged. A bystander, a former victim of the corrupt system, intervened, and in a fit of rage, he attacked the oppressors, sparking a riot. The girl was released, and the oppressors were punished.


In [110]:
# No Steering 
text_baseline = generateSteering(
    user_text=prompt,
    system_text= system_prompt,
    model=model,
    steering_vector = None,
    tokenizer=tokenizer,
    # steering_strength=0.8
)

print(text_baseline)

As the sun set over the small village, a group of friends gathered for a picnic. But when Maria, a newcomer, was excluded from the seating arrangement, her friend Emma's anger flared up. "Why didn't you include me?" Maria asked, her voice trembling. Emma's defensiveness and refusal to apologize sparked a heated argument, shattering the peaceful evening.


In [111]:
id_prompt = "Berikanlah cerita tentang kemarahan yang muncul dari ketidakpercayaan, pengkhianatan, atau rasa tidak dihargai"

In [112]:
text_baseline = generateSteering(
    user_text=id_prompt,
    system_text= system_prompt,
    model=model,
    steering_vector = steering_vectors['anger'],
    tokenizer=tokenizer,
    steering_strength=0.9
)

print(text_baseline)

Applied steering to layers [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31] with strength 0.9
Seorang raja yang egois dan tidak adil memanfaatkan kekuasaannya untuk menganiaya rakyatnya. Ketika rakyatnya menentang, raja tersebut menuduh mereka sebagai pemberontak dan menghukum mereka dengan kekerasan. Rakyat tersebut marah dan mengangkangkan raja tersebut, mengakibatkan kerusuhan dan kekacauan.


In [ ]:


text_baseline = generateSteering(
    user_text=id_prompt,
    system_text= system_prompt,
    model=model,
    # steering_vector = steering_vectors['anger'],
    tokenizer=tokenizer,
    # steering_strength=0.9
)
print(text_baseline)

Cerita tentang Raka dan Ayahnya

Raka adalah seorang anak yang suka bermain di hutan. Suatu hari, ayahnya, seorang petani, meminta Raka untuk menjaga ladang jagungnya. Namun, Raka malah pergi bermain dengan temannya dan lupa akan tugasnya. Ketika ayahnya mengetahui hal ini, ia marah dan memarahi Raka.

Raka merasa kesal dan tidak percaya diri. Ia merasa ayahnya tidak menghargai usahanya dan tidak percaya diri untuk menjaga ladang. Raka menjadi lebih keras kepala dan enggan untuk mendengarkan nasihat ayahnya.

Namun, suatu hari, hujan turun dan ladang jagung ayah Raka terkena banjir. Raka harus berlari ke arah ladang untuk menghemat s
